# Quand l'ordinateur se trompe

| Module | Durée | Prérequis |
|---|---|---|
| M2 – Systèmes linéaires | 75 min | M1 |

In [ ]:
# Style graphique Liouaï : couleurs du logo, axes sobres
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

LIOUAI = {"navy": "#243743", "vert": "#28B78D", "vert_texte": "#167A5C", "gris": "#A1A9AE",
          "gris_texte": "#5C6870", "jaune": "#E5C800", "fond": "#F3F5F6"}
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[LIOUAI["vert"], LIOUAI["navy"], LIOUAI["gris"],
                                         LIOUAI["jaune"], LIOUAI["vert_texte"], LIOUAI["gris_texte"]]),
    "axes.edgecolor": LIOUAI["gris"], "axes.labelcolor": LIOUAI["navy"],
    "axes.titlecolor": LIOUAI["navy"], "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": LIOUAI["gris_texte"], "ytick.color": LIOUAI["gris_texte"],
    "text.color": LIOUAI["navy"], "grid.color": "#E1E6E9", "legend.frameon": False,
    "font.family": "sans-serif", "font.sans-serif": ["Roboto", "Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi": 100,
})
# Dégradé de marque : gris clair -> vert -> navy (et sa version inversée)
if "liouai" not in mpl.colormaps:
    _cmap = LinearSegmentedColormap.from_list("liouai", [LIOUAI["fond"], LIOUAI["vert"], LIOUAI["navy"]])
    mpl.colormaps.register(_cmap)
    mpl.colormaps.register(_cmap.reversed())

## ① Définir la question

Un laboratoire de recyclage reçoit un lingot de 100 g de laiton. Il sait que ce lingot a été obtenu en fondant ensemble deux laitons connus :

- le laiton **A** contient 60 % de cuivre ;
- le laiton **B** contient 61 % de cuivre.

Une analyse mesure **60,5 g de cuivre** dans le lingot. La balance d'analyse est précise à ± 0,1 g.

**Question du laboratoire : quelle masse de chaque laiton contient le lingot, et peut-on faire confiance à la réponse ?**

**Activité** : Quelles informations sont utiles ? Quelle information inquiète, avant même de calculer ?

Indiquez vos réponses et justifiez-les.

## ② Estimer avant de calculer

Si la mesure de cuivre était 60,6 g au lieu de 60,5 g (une erreur de 0,1 g, soit 0,17 %), de combien pensez-vous que la masse de laiton B changerait ?

In [ ]:
pari_variation_B = 1.0   # en grammes (modifiez)

## ③ Abstraire

Même modèle que le bijoutier du notebook 2.1, avec $x$ et $y$ les masses de laiton A et B :

$$
\begin{cases}
x + y = 100 \\
0{,}60\,x + 0{,}61\,y = 60{,}5
\end{cases}
$$

Géométriquement, ce sont deux droites **presque parallèles** : les deux laitons se ressemblent beaucoup. Leur point d'intersection est donc très mal défini.

**Le concept : le conditionnement.** Le **nombre de conditionnement** d'une matrice, noté $\operatorname{cond}(A)$, mesure à quel point une petite erreur $\Delta\mathbf{b}$ sur les données peut être amplifiée en une erreur $\Delta\mathbf{x}$ sur la solution :

$$
\underbrace{\frac{\lVert \Delta\mathbf{x} \rVert}{\lVert \mathbf{x} \rVert}}_{\text{erreur relative sur la solution}}
\;\le\; \operatorname{cond}(A) \times
\underbrace{\frac{\lVert \Delta\mathbf{b} \rVert}{\lVert \mathbf{b} \rVert}}_{\text{erreur relative sur les données}}
$$

Plus $\operatorname{cond}(A)$ est grand, plus le problème est **mal conditionné**. Ce n'est pas un défaut de l'ordinateur : c'est une propriété du problème lui-même.

## ④ Calculer

### Les laitons

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

A = np.array([[1, 1], [0.60, 0.61]])

for cuivre in [60.4, 60.5, 60.6]:
    x, y = np.linalg.solve(A, [100, cuivre])
    print(f"cuivre mesuré {cuivre} g  ->  laiton A {x:6.1f} g, laiton B {y:6.1f} g")
print(f"\ncond(A) = {np.linalg.cond(A):.0f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
xs = np.linspace(0, 100, 201)
styles = [(60.4, ":"), (60.5, "-"), (60.6, "--")]

# À gauche : les droites elles-mêmes, presque indiscernables
ax = axes[0]
ax.plot(xs, 100 - xs, lw=2.5, label="x + y = 100")
for cuivre, style in styles:
    ax.plot(xs, (cuivre - 0.60 * xs) / 0.61, style, lw=1.8, label=f"0,60 x + 0,61 y = {cuivre}")
ax.set_xlabel("x : laiton A (g)"); ax.set_ylabel("y : laiton B (g)")
ax.set_title("Les droites : presque confondues"); ax.legend(fontsize=8); ax.grid(alpha=0.3)

# À droite : l'écart vertical entre chaque droite du cuivre et la droite de masse.
# L'intersection est l'endroit où cet écart s'annule.
ax = axes[1]
for cuivre, style in styles:
    ecart = (cuivre - 0.60 * xs) / 0.61 - (100 - xs)
    x_i = np.linalg.solve(A, [100, cuivre])[0]
    ax.plot(xs, ecart, style, lw=2, label=f"cuivre = {cuivre} g")
    ax.plot(x_i, 0, "o", color=LIOUAI["navy"], ms=7)
    ax.annotate(f"{x_i:.0f}", (x_i, 0), textcoords="offset points", xytext=(0, -16), ha="center", fontsize=9)
ax.axhline(0, color=LIOUAI["navy"], lw=0.8)
ax.set_xlabel("x : laiton A (g)"); ax.set_ylabel("écart vertical entre les droites (g)")
ax.set_title("L'écart s'annule à l'intersection"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.suptitle("Une erreur de 0,1 g sur le cuivre déplace l'intersection de 10 g")
plt.tight_layout(); plt.show()

**Activité** : Sur le graphique de droite, l'écart entre les droites varie très lentement avec x (la pente est faible). Pourquoi est-ce exactement ce qui rend l'intersection si sensible à la mesure ?

Indiquez vos réponses et justifiez-les.

### Les nombres de l'ordinateur ne sont pas les nombres réels

Un ordinateur représente un nombre à virgule avec environ 16 chiffres significatifs. La plupart des nombres décimaux, même simples, ne sont pas représentables exactement.

In [ ]:
print("0.1 + 0.2 == 0.3 ?", 0.1 + 0.2 == 0.3)
print("0.1 + 0.2 =", f"{0.1 + 0.2:.20f}")
print("Précision relative de la machine (epsilon) :", np.finfo(float).eps)

Ces erreurs d'arrondi minuscules (environ $10^{-16}$) sont elles aussi des « erreurs sur les données ». Multipliées par $\operatorname{cond}(A)$, elles deviennent visibles quand le conditionnement est très grand.

**Règle pratique.** On perd environ $\log_{10}\operatorname{cond}(A)$ chiffres significatifs. Avec 16 chiffres au départ, il en reste à peu près

$$
16 - \log_{10} \operatorname{cond}(A).
$$

### Une famille de matrices célèbres : les matrices de Hilbert

La matrice de Hilbert de taille $n$ a pour coefficients

$$
h_{ij} = \frac{1}{i + j - 1}, \qquad \text{par exemple} \quad H_3 = \begin{pmatrix} 1 & \frac12 & \frac13 \\ \frac12 & \frac13 & \frac14 \\ \frac13 & \frac14 & \frac15 \end{pmatrix}.
$$

Elle apparaît quand on approche une fonction par un polynôme. On résout $H\mathbf{x} = \mathbf{b}$, avec $\mathbf{b}$ choisi pour que la solution exacte soit $(1, 1, \dots, 1)$, et on mesure l'erreur.

In [ ]:
from scipy.linalg import hilbert

tailles = range(2, 15)
conds, erreurs = [], []
for n in tailles:
    H = hilbert(n)
    x_exact = np.ones(n)
    x_calcule = np.linalg.solve(H, H @ x_exact)
    conds.append(np.linalg.cond(H))
    erreurs.append(np.abs(x_calcule - x_exact).max())

for n, c, e in zip(tailles, conds, erreurs):
    chiffres = max(0, 16 - np.log10(c))
    print(f"n = {n:2d}   cond = {c:9.1e}   erreur max = {e:9.1e}   chiffres fiables prévus ≈ {chiffres:4.1f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(conds, erreurs, "o-", label="erreur mesurée")
c = np.array(conds)
ax.loglog(c, c * np.finfo(float).eps, "--", label="cond × epsilon")
ax.set_xlabel("conditionnement cond(H)"); ax.set_ylabel("erreur maximale sur x")
ax.set_title("Matrices de Hilbert : l'erreur suit le conditionnement"); ax.legend(); ax.grid(alpha=0.3, which="both")
plt.show()

### Une erreur de méthode : le pivot trop petit

Le conditionnement n'est pas la seule source d'erreur. La **méthode** compte aussi. Prenons un système parfaitement bien conditionné :

$$
\begin{cases}
10^{-20}\,x + y = 1 \\
x + y = 2
\end{cases}
$$

Sa solution est très proche de $x = 1$, $y = 1$. Appliquons le pivot de Gauss « naïf », qui prend toujours le premier coefficient comme pivot, même minuscule.

In [ ]:
def gauss_naif(A, b):
    # Pivot de Gauss sans échange de lignes, en nombres à virgule
    M = np.column_stack([np.array(A, dtype=float), np.array(b, dtype=float)])
    n = len(M)
    for col in range(n):
        for i in range(col + 1, n):
            M[i] -= (M[i, col] / M[col, col]) * M[col]
    x = np.zeros(n)
    for i in reversed(range(n)):
        x[i] = (M[i, n] - M[i, i + 1:n] @ x[i + 1:]) / M[i, i]
    return x

A_piege = [[1e-20, 1], [1, 1]]
b_piege = [1, 2]
print("Pivot naïf        :", gauss_naif(A_piege, b_piege))
print("np.linalg.solve   :", np.linalg.solve(A_piege, b_piege))
print("cond(A)           :", f"{np.linalg.cond(A_piege):.2f}")

Le pivot naïf trouve $x = 0$ au lieu de $x \approx 1$. Le problème n'est pas mal conditionné : c'est la **méthode** qui est instable. En divisant par $10^{-20}$, elle crée des nombres énormes qui écrasent les autres. `np.linalg.solve` utilise le **pivot partiel** : à chaque étape, il choisit comme pivot le plus grand coefficient de la colonne, en valeur absolue. Les fonctions `pivot_de_gauss` et `gauss_flottants` du notebook 2.2 le font aussi.

## ⑤ Interpréter

In [ ]:
y1 = np.linalg.solve(A, [100, 60.5])[1]
y2 = np.linalg.solve(A, [100, 60.6])[1]
print(f"Votre pari : {pari_variation_B} g ; variation réelle de la masse de laiton B : {y2 - y1:.1f} g")

**Activité** :

1. Répondez au laboratoire : avec une balance à ± 0,1 g, entre quelles valeurs se trouve la masse de laiton B ? La réponse est-elle utile ?
2. Le calcul de l'ordinateur était-il faux pour les laitons ? Où se trouve la vraie source d'erreur ?
3. À partir de quelle taille de matrice de Hilbert le résultat ne contient-il plus aucun chiffre fiable ? La règle des « $16 - \log_{10}\operatorname{cond}(A)$ chiffres » le prévoyait-elle ?

Indiquez vos réponses et justifiez-les.

## ⑥ Critiquer et généraliser

- **Classer les sources d'erreur.** Pour chacune des trois situations du notebook, dites si l'erreur vient : des **données** (mesure), de la **représentation** des nombres en machine, ou de la **méthode** de calcul.

| Situation | Source principale de l'erreur | Remède possible |
|---|---|---|
| laitons voisins | … | … |
| matrices de Hilbert | … | … |
| pivot minuscule | … | … |

- **Évaluer la faisabilité avant de calculer.** Que proposer au laboratoire pour obtenir une réponse fiable ? Mesurer le zinc n'apporterait rien : dans un laiton, zinc = 100 % − cuivre, c'est la même information. Montrez que l'incertitude sur la masse de B vaut

$$
\Delta y = \frac{\Delta m_{\text{mesure}}}{\text{écart de teneur entre A et B}}.
$$

 Quels sont alors les deux leviers pour améliorer la réponse ?
- **Généraliser.** Écrivez une fonction `fiabilite(A)` qui renvoie $\operatorname{cond}(A)$ et le nombre de chiffres fiables estimé, et qui affiche un avertissement au-delà d'un seuil que vous choisirez.

In [ ]:
def fiabilite(A, seuil_chiffres=4):
    # À compléter
    ...

## Corrigé

*À n'ouvrir qu'après avoir cherché.*

In [ ]:
def fiabilite(A, seuil_chiffres=4):
    c = np.linalg.cond(A)
    chiffres = max(0.0, 16 - np.log10(c))
    if chiffres < seuil_chiffres:
        print(f"Attention : cond = {c:.1e}, il reste environ {chiffres:.0f} chiffre(s) fiable(s) "
              "(sans compter les erreurs de mesure, amplifiées elles aussi).")
    return c, chiffres

fiabilite(A)
fiabilite(hilbert(10))
print("Bouquets du notebook 2.2 :", fiabilite(np.array([[3, 2, 1], [1, 4, 2], [2, 1, 3]])))

# Incertitude sur la masse de B = incertitude de mesure / écart de teneur entre les deux laitons
for element, ecart in [("cuivre (60 % contre 61 %)", 0.01), ("plomb (0 % contre 2 %)", 0.02)]:
    for precision in [0.1, 0.01]:
        print(f"{element:26s} balance à ± {precision} g  ->  laiton B connu à ± {precision / ecart:4.1f} g")
print("Deux leviers : une mesure plus précise, ou un élément dont la teneur diffère davantage entre A et B.")

## Bilan — auto-évaluation

Pour chaque compétence travaillée dans ce notebook, choisissez un niveau : **0** non observé · **1** guidé · **2** autonome · **3** transfert.
La cellule enregistre votre bilan dans `progression.json`, à la racine du site.

In [ ]:
import json, datetime, pathlib

bilan = {
    "Lister les sources d'erreur liées aux limites du calcul": 1,
    "Évaluer la faisabilité d'obtenir une réponse utile": 1,
    "Identifier erreurs systématiques et aléatoires": 1,
    "Sentir la fiabilité d'un modèle": 1,
    "Sentir si le résultat est plausible pour le contexte": 1,
}   # modifiez les niveaux

ici = pathlib.Path.cwd()
racine = next((p for p in [ici, *ici.parents] if (p / "myst.yml").exists()), ici)
fichier = racine / "progression.json"
historique = json.loads(fichier.read_text(encoding="utf-8")) if fichier.exists() else []
historique.append({"notebook": "2.4-quand-l-ordinateur-se-trompe", "date": datetime.date.today().isoformat(), "niveaux": bilan})
fichier.write_text(json.dumps(historique, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Bilan enregistré ({len(historique)} entrée(s) dans {fichier.name}).")